In [ ]:
import kagglehub
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torch.optim import Adam
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torchvision.transforms import ToTensor, Compose, Normalize

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
%pip install kagglehub catboost lightgbm tqdm -q
from catboost import CatBoostClassifier


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
anonymized_path = os.path.join(path, 'Q3_data.csv')
df_anon= pd.read_csv(anonymized_path)


In [ ]:
# Task 2: Write your code here:
df_anon.head()

In [ ]:
# Task 3: Write your code here:
df_anon.info()

In [ ]:
# Task 4: Write your code here:
df_anon.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df_anon):
  missing_values = df_anon.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nWe need to handle missing values.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_anon)

In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_anon.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_anon.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")



In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df_anon.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    #  Apply fit_transform to encode the column
    df_anon[col] = le.fit_transform(df_anon[col])

df_anon.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols =  df_anon.select_dtypes(include=["number"]).columns.drop("Target")

scaler = StandardScaler()

# Apply fit_transform to scale the numerical columns
df_anon[numerical_cols] = scaler.fit_transform(df_anon[numerical_cols])

df_anon.head()

In [ ]:
# Task 5: Write your code here:
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.countplot(data=df_anon, x='Target')
plt.title('Distribution of Target Variable')
plt.xlabel('Target')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task 1: Write your code here:
X = df_anon.drop("Target",axis=1)
y = df_anon['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}
n_splits = 5
skf = StratifiedKFold(n_splits= 5, shuffle=True, random_state=42)

results = {}

for model_name in models:
  results[model_name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}




for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # Fit the model on train data
    model.fit(X_train, y_train)

    # Use the model to predict the test data
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    results[model_name]['accuracy'].append(accuracy)
    results[model_name]['f1'].append(f1)


    for model_name in results:
      print(f"\n{model_name}:")
      # Print the average of each evaluation metric across folds
      print(f"  Accuracy:  {np.mean(results[model_name]['accuracy']):.4f}")
      print(f"  F1-Score:  {np.mean(results[model_name]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
importances = {}

importances['CatBoost'] = models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: